# 03 Dynamic Hourly Anchor Selection

This notebook discovers the hourly anchor candidates for the downstream 15-minute extension by reusing the existing hourly scenario-generation selection logic.

Current scope:

- discover the latest eligible hourly benchmark candidates from saved artifacts;
- apply the existing hourly scenario-selection rules without hardcoding candidate names;
- identify the deterministic winner and the hour-ranking winner, or accept a single model if the hourly logic returns one winner for both roles;
- save the selected hourly anchor provenance for later quarter-hour forecasting phases.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path
    for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]
    if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)

PACKAGE_ROOT = REPO_ROOT / "scripts" / "Data" / "02_Forecasting" / "01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from quarterhour_da import (
    QuarterHourDAExtensionConfig,
    assert_thesis_grade_actual_source_authorized,
    build_thesis_grade_frozen_actual_metadata,
    find_frozen_actual_version,
    find_latest_canonical_actual_run,
    find_latest_observed_deterministic_run,
    find_latest_phase01_run,
    find_latest_phase02_run,
    find_latest_phase03_run,
    find_latest_phase04_run,
    find_latest_phase07_run,
    find_latest_phase07_upstream_refresh_run,
    load_frozen_actual_diagnostics,
    load_frozen_actual_manifest,
    load_frozen_actual_path,
    resolve_frozen_actual_registry_entry,
    run_observed_market_deterministic_forecast,
)

config = QuarterHourDAExtensionConfig()
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
plt.style.use("seaborn-v0_8-whitegrid")

## Optional Phase 3 Runner

The notebook loads the latest saved Phase 3 artifact by default. Set `RUN_PHASE03 = True` only when you want to rerun dynamic hourly anchor discovery from inside the notebook.

In [ ]:
RUN_PHASE03 = False

if RUN_PHASE03:
    command = [
        sys.executable,
        str(REPO_ROOT / "scripts" / "Data" / "02_Forecasting" / "01_DA_prices" / "run_15min_phase03_anchor_selection.py"),
    ]
    completed = subprocess.run(command, cwd=REPO_ROOT, capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f"Phase 3 anchor selection failed with exit code {completed.returncode}.")

In [ ]:
latest_run = find_latest_phase03_run(config)
if latest_run is None:
    raise FileNotFoundError("No saved Phase 3 artifact exists yet. Run the phase 3 script first.")

selection_contract = pd.read_csv(latest_run / "selection_contract.csv")
candidate_availability = pd.read_csv(latest_run / "candidate_availability.csv")
ignored_runs = pd.read_csv(latest_run / "ignored_runs.csv")
evaluation_slice_summary = pd.read_csv(latest_run / "evaluation_slice_summary.csv")
official_naive_selected = pd.read_csv(latest_run / "official_naive_selected.csv")
official_naive_summary = pd.read_csv(latest_run / "official_naive_summary.csv")
selection_summary = pd.read_csv(latest_run / "candidate_selection_summary.csv")
selected_anchor_models = pd.read_csv(latest_run / "selected_anchor_models.csv")
runtime_summary = pd.read_csv(latest_run / "selected_candidate_runtime_summary.csv")
model_settings_summary = pd.read_csv(latest_run / "selected_candidate_model_settings_summary.csv")
run_summary = json.loads((latest_run / "run_summary.json").read_text(encoding="utf-8"))

display(pd.DataFrame([{"latest_phase03_run": str(latest_run)}]))

## Selection Contract

In [ ]:
display(selection_contract)

## Evaluation Slice Used By The Hourly Selection Logic

In [ ]:
display(evaluation_slice_summary)

## Official Naive Benchmark Used For rMAE

In [ ]:
display(official_naive_selected)

## Selected Hourly Anchor Models

In [ ]:
display(selected_anchor_models)

## Candidate Comparison Plot

Lower `rMAE` is better for the deterministic role. Higher `ranking_score` is better for the hour-ranking role.

In [ ]:
plot_frame = selection_summary.copy()
selected_keys = set(selected_anchor_models["candidate_key"].astype(str).tolist())

fig, ax = plt.subplots(figsize=(8.5, 5.5))
for _, row in plot_frame.iterrows():
    candidate_key = str(row["candidate_key"])
    is_selected = candidate_key in selected_keys
    ax.scatter(
        row["rmae"],
        row["ranking_score"],
        s=90 if is_selected else 45,
        alpha=0.95 if is_selected else 0.65,
    )
    if is_selected:
        ax.annotate(str(row["candidate_label"]), (row["rmae"], row["ranking_score"]), xytext=(6, 4), textcoords="offset points")

ax.set_xlabel("rMAE (lower is better)")
ax.set_ylabel("Ranking score (higher is better)")
ax.set_title("Hourly Candidate Selection View")
plt.show()

## Full Candidate Selection Summary

In [ ]:
display(selection_summary.sort_values(['rmae', 'ranking_score'], ascending=[True, False]))

## Candidate Availability

In [ ]:
display(candidate_availability)

## Selected Candidate Runtime Summary

In [ ]:
display(runtime_summary)

## Selected Candidate Model Settings Summary

In [ ]:
display(model_settings_summary)

## Ignored Runs

In [ ]:
display(ignored_runs)